* experimental data
* modeling on transmon circuit
* identification of inaccuracies
* modification to transmon circuit
* improvement on modeling result

In [1]:
import pandas
from des_scq import models
from des_scq.circuit import Charge
from des_scq.optimization import Optimization
from torch import tensor
import torch
MSE = torch.nn.MSELoss()

from numpy import arange,polyfit
from des_scq.utils import plotCompare,plotCombine
from multiprocess import Pool
from des_scq.discovery import uniformParameters,initializationSequential
import pickle
from torch import set_num_threads
set_num_threads(30)

### data loading

In [2]:
dExp = pandas.read_csv('./dRamsey.csv',dtype={'ref':float,'f':float,'transition':str})

In [3]:
dExp[['ref','f']] *= 1e-9

In [4]:
dExp.head()

,ref,f,transition
0,4.614254,4.616248,01
1,4.614232,4.614175,01
2,4.613703,4.613688,01
3,4.613815,4.613790,01
4,4.613511,4.613604,01


In [5]:
dI = dExp[dExp['transition']=='01']
dII = dExp[dExp['transition']=='12']
dIII = dExp[dExp['transition']=='02']

In [6]:
plot = {'01':(dI['ref'],dI['f'])}
plot['12'] = (dII['ref'],dII['f'])
plot['02'] = (dIII['ref'],dIII['f'])

In [7]:
plotCombine(plot,'Experimental Transition Frequencies','reference','transition',mode='markers')

### transition energies

In [8]:
def transitionEnergy(df,transition='01'):
    # For one transition, e.g. '01'
    mask = df['transition'] == transition
    ref = df[mask]['ref'].values
    f   = df[mask]['f'].values

    coeffs, cov = polyfit(ref, f, deg=1, cov=True)
    slope, intercept = coeffs

    f_transition = -intercept / slope
    return f_transition

In [9]:
e01 = tensor(transitionEnergy(dExp,'01'))
e12 = tensor(transitionEnergy(dExp,'12'))
e02 = tensor(transitionEnergy(dExp,'02'))
print(e01,e12,e02)

tensor(-0.0002, dtype=torch.float64) tensor(0.4608, dtype=torch.float64) tensor(9.6944, dtype=torch.float64)


### circuit initialization

In [10]:
basis = [256]
circuit = models.transmon(Charge,basis)
print(circuit.circuitComponents())
parameters = uniformParameters(circuit,('J','C'),5,5)

{'J': 10.000000000000004, 'C': 1.6666666666666659}


### loss function

In [11]:
def lossParameterEstimation(Spectrum,flux_profile):
    loss = tensor(0.0)
    for index,(spectrum,state) in enumerate(Spectrum):
        loss += MSE(spectrum[1]-spectrum[0],e01)
        loss += MSE(spectrum[2]-spectrum[1],e12)
        loss += MSE(spectrum[2]-spectrum[0],e02)
    loss = loss
    return loss,dict()

### optimization

In [12]:
iterations = 100
lr = .001
flux_profile = [dict()]
optim = Optimization(circuit,flux_profile,loss_function=lossParameterEstimation)

In [15]:
Search = initializationSequential(parameters,optim,iterations=100,lr=.005)

0 --------------------
1 --------------------
2 --------------------
3 --------------------
4 --------------------


### analysis

In [16]:
Losses = dict()
for index,(parameter,(dLogs,dParams,dCircuit)) in enumerate(zip(parameters,Search)):
    loss = dLogs['loss'].to_numpy()
    if loss[0]-loss[-1] > 0:       
        Losses[str(index)] = loss

In [17]:
plotCompare(arange(iterations),Losses)